# Sweep Mode C — Post-build hot-swap re-solve

**Mode C (resolve)** builds the model **once** and then re-solves it many times, swapping parameter values in place. It is the fastest sweep because it skips the rebuild, but it only changes *mutable* parameters:

`pDemandElec`, `pENSCost`, `pLinearVarCost`, `pEFOR`, `pReserveMargin`, `pRESEnergy`.

For anything structural (network topology, new candidate units) use [Mode A](5.1-Sweep-Mode-A-PreBuild.ipynb) or [Mode B](5.2-Sweep-Mode-B-InMemory.ipynb), which rebuild the model.

## Build the model once

We build the 9n model with `openTEPES_run` and keep the returned model object. The build is the slow step; we pay it only once.

In [1]:
import os, shutil
import pandas as pd

def coarse_copy(src, dst):
    """Copy a case folder and set a coarse time resolution so the example runs fast."""
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    p = os.path.join(dst, "oT_Data_Parameter_9n.csv")
    df = pd.read_csv(p)
    df.loc[:, "TimeStep"] = 24   # coarse resolution, just to keep this tutorial quick
    df.to_csv(p, index=False)

coarse_copy("9n", "sweepC/9n")

from openTEPES.openTEPES import openTEPES_run
model = openTEPES_run("sweepC", "9n", "appsi_highs", "No", "No")

Input data                             ****
Reading the CSV files                  ...  0 s
Reading    input data                  ...  0 s


Setting up input data                  ...  1 s
Setting up variables                   ...  0 s
Total cost o.f.      model formulation ****
Investment elec      model formulation ****
Period 2030, Scenario sc01, Stage st1
Generation oper o.f. model formulation ****


Investment & operation var constraints ****
Inertia, oper resr, demand constraints ****
Storage   scheduling       constraints ****
Unit commitment            constraints ****


Ramp and min up/down time  constraints ****
Network    switching model constraints ****
Network    operation model constraints ****
Problem solving                        #### 1


Termination condition:  optimal
  Total system                 cost [MEUR]  159.2111765517775  Constraints 41136  Variables 50966  Seconds 4
***** Period: 2030, Scenario: sc01, Stage: st1 ******
  Total generation  investment cost [MEUR]  0
  Total generation  retirement cost [MEUR]  0
  Total reservoir   investment cost [MEUR]  0.0
  Total network     investment cost [MEUR]  4.041225328215885
  Total H2   pipe   investment cost [MEUR]  0.0
  Total heat pipe   investment cost [MEUR]  0.0
  Total generation  operation  cost [MEUR]  155.16550622161176
  Total consumption operation  cost [MEUR]  0.002853297506587093
  Total emission               cost [MEUR]  0.0
  Total network losses penalty cost [MEUR]  0.001591704443277054
  Total reliability electr     cost [MEUR]  0.0
Writing            investment results  ...  0 s
Writing          cost summary results  ...  0 s
Writing           KPI summary results  ...  0 s


Writing elect network summary results  ...  0 s


Writing              economic results  ...  0 s


## Re-solve under a demand sweep

`overlay_scaled(model, "pDemandElec", 1.10)` builds an overlay that scales the current demand by 10%. `resolve` re-solves the model once per overlay. Each overlay applies relative to the baseline, not on top of the previous one, so an empty overlay `{}` always reproduces the baseline.

`resolve` needs a non-persistent solver, so we pass `"highs"` here (the rest of the tutorial uses `"appsi_highs"`).

In [2]:
from openTEPES.openTEPES_ProblemSolvingResolve import resolve, overlay_scaled

overlays = [
    {},                                          # baseline (100% demand)
    overlay_scaled(model, "pDemandElec", 1.10),  # +10%
    overlay_scaled(model, "pDemandElec", 1.20),  # +20%
]

results = resolve(model, "highs", overlays)

In [3]:
df = pd.DataFrame(results)
df["demand"] = ["100%", "110%", "120%"]
df[["demand", "status", "total_cost_meur"]]

,demand,status,total_cost_meur
0,100%,optimal,159.211177
1,110%,optimal,200.039066
2,120%,optimal,243.202104


## Choosing a sweep mode

| Mode | Reads input | Builds model | Best when |
|---|---|---|---|
| A pre-build | per case | per case | cases are different folders / scenarios |
| B in-memory | once | per case | cases share inputs, differ by a parameter |
| C resolve | once | once | many re-solves of the mutable parameters above |

Start with Mode A when in doubt. Move to B when your cases share inputs, and to C when you re-solve the same model many times over mutable parameters like demand or costs.